In [1]:
%pip -q install --upgrade "sagemaker>=2,<3" boto3 botocore

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
skops 0.14.0 requires prettytable>=3.9, which is not installed.
aiobotocore 3.8.0 requires botocore<1.43.47,>=1.43.3, but you have botocore 1.43.78 which is incompatible.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 21.0.0 which is incompatible.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.6.0 which is incompatible.
sagemaker-mlops 1.12.0 requires sagemaker-core>=2.12.0, but you have sagemaker-core 1.0.78 which is

Note: you may need to restart the kernel to use updated packages.


In [26]:
# Standard Library Imports
import os

# 1. Suppress the SageMaker v2 deprecation warning
os.environ["SAGEMAKER_SUPPRESS_V2_WARNING"] = "1"

# Standard Library Imports
import json
import time
from pathlib import Path


# Data & ML Libraries
import numpy as np
import pandas as pd


# Cloud & MLOps
import boto3
import botocore


# AWS SageMaker SDK
import sagemaker

from sagemaker.inputs import TrainingInput
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.parameters import (
    ParameterString,
    ParameterInteger,
    ParameterFloat,
)
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.step_collections import RegisterModel


print(f"SageMaker Version: {sagemaker.__version__}")

SageMaker Version: 2.257.6


In [5]:
# ------------------------------------------------------------
# Student/team configuration
# ------------------------------------------------------------
REGION = "ap-southeast-1"
SEMESTER = "26S1"

TEAM_ID = "team09"
STUDENT_ID = "s901"

BUCKET = "nyp-26s1-iti113"
ROLE_ARN = "arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team09"

# Separate pipeline for the S3-triggered workflow
TRIGGERED_PIPELINE_NAME = f"iti113-{TEAM_ID}-bank-fraud-detection-triggered"

# S3 prefixes
TEAM_PREFIX = f"iti113/{TEAM_ID}/triggered-pipeline"
TRIGGER_PREFIX = f"iti113/{TEAM_ID}/trigger/input"
MANUAL_TEST_PREFIX = f"iti113/{TEAM_ID}/manual-input"

# Model registry group for this triggered workflow
MODEL_PACKAGE_GROUP_NAME = f"{TEAM_ID}-BankFraudDetection-Triggered"

# SageMaker instance choices
PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE = "ml.m5.large"

# Local synthetic dataset filename
LOCAL_FRAUD_FILE = "bank_fraud.csv"

# Default input path used if no parameter is passed.
# Lambda/manual tests will override this using the InputDataUrl pipeline parameter.
DEFAULT_INPUT_DATA_URL = (
    f"s3://{BUCKET}/iti113/{TEAM_ID}/data/bank-fraud-detection/raw/bank_fraud.csv"
)

# Keep SageMaker SDK-generated code/model artifacts inside the team S3 prefix.
TRAINING_OUTPUT_PREFIX = f"s3://{BUCKET}/{TEAM_PREFIX}/training-output"
TRAINING_CODE_LOCATION = f"s3://{BUCKET}/{TEAM_PREFIX}/code/training"

boto_session = boto3.Session(region_name=REGION)

sagemaker_session = sagemaker.Session(
    boto_session=boto_session,
    default_bucket=BUCKET,
    default_bucket_prefix=TEAM_PREFIX
)

pipeline_session = PipelineSession(
    boto_session=boto_session,
    sagemaker_client=boto_session.client("sagemaker"),
    default_bucket=BUCKET,
    default_bucket_prefix=TEAM_PREFIX
)

s3 = boto3.client("s3", region_name=REGION)
sm = boto3.client("sagemaker", region_name=REGION)

print("Region:", REGION)
print("Bucket:", BUCKET)
print("Team prefix:", TEAM_PREFIX)
print("Trigger prefix:", TRIGGER_PREFIX)
print("Role:", ROLE_ARN)
print("Triggered pipeline:", TRIGGERED_PIPELINE_NAME)
print("Model package group:", MODEL_PACKAGE_GROUP_NAME)

Region: ap-southeast-1
Bucket: nyp-26s1-iti113
Team prefix: iti113/team09/triggered-pipeline
Trigger prefix: iti113/team09/trigger/input
Role: arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team09
Triggered pipeline: iti113-team09-bank-fraud-detection-triggered
Model package group: team09-BankFraudDetection-Triggered


In [6]:
# Define pipeline parameters
input_data_url_param = ParameterString(
    name="InputDataUrl",
    default_value=DEFAULT_INPUT_DATA_URL
)

n_estimators_param = ParameterInteger(
    name="NEstimators",
    default_value=500
)

max_depth_param = ParameterInteger(
    name="MaxDepth",
    default_value=8
)

quality_gate_auc_param = ParameterFloat(
    name="QualityGateAUC",
    default_value=0.70
)

print("Pipeline parameters created:")
print("- InputDataUrl")
print("- NEstimators")
print("- MaxDepth")
print("- QualityGateAUC")

Pipeline parameters created:
- InputDataUrl
- NEstimators
- MaxDepth
- QualityGateAUC


In [8]:
# Create local source folder
SRC_DIR = Path("triggered_pipeline_src")
SRC_DIR.mkdir(parents=True, exist_ok=True)

# The SageMaker SKLearn container does not include
# xgboost or imbalanced-learn. A requirements.txt is
# auto-installed before train.py runs
with open(SRC_DIR / "requirements.txt", "w") as f:
    f.write("xgboost>=2.0,<3\n")
    f.write("imbalanced-learn>=0.11\n")
    f.write("scikit-learn>=1.2.0\n")

print("Source folder created:")
print(SRC_DIR.resolve())

Source folder created:
/home/sagemaker-user/ITI113_project/triggered_pipeline_src


**Insight — this pipeline's `preprocess.py` reuses the identical 31-numeric/14-categorical feature-engineering logic as Notebook 03**, preserving the same feature contract across both the original and enhanced pipelines so a model trained by either one is comparable and interchangeable at the Model Registry level.

In [15]:
%%writefile triggered_pipeline_src/preprocess.py
import argparse
import glob
import logging
import os

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

TARGET_COLUMN = "is_fraud"

NUMERIC_FEATURES = [
    "hour_of_day",
    "customer_age",
    "credit_score",
    "account_age_years",
    "account_balance",
    "transaction_amount",
    "num_prev_transactions",
    "transaction_freq_monthly",
    "distance_from_home_km",
    "time_since_last_txn_hrs",
    "failed_attempts",
    "log_transaction_amount",
    "log_account_balance",
    "amount_to_balance_ratio",
    "risk_score",
    "failed_x_transaction",
    "night_x_international",
    "pin_x_failed",
    "night_x_pin",
    "amount_x_distance",
    "amount_x_failed",
    "risk_x_amount",
    "risk_x_distance",
    "customer_mean_amount",
    "customer_std_amount",
    "customer_mean_distance",
    "customer_std_distance",
    "customer_txn_count",
    "customer_mean_balance",
    "amount_zscore",
    "distance_zscore",
    "balance_deviation",
]
CATEGORICAL_FEATURES = [
    "is_weekend",
    "is_night_transaction",
    "country",
    "city",
    "merchant_category",
    "payment_method",
    "device_type",
    "is_international",
    "pin_changed_recently",
    "customer_age_group",
    "hour_bin",
    "high_amount",
    "far_from_home",
    "rapid_txn",
]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

DROP_COLS = [
    "transaction_id",
    "customer_id",
    "transaction_date",
    "transaction_time",
    "fraud_type",
]

# Defaults for raw transaction fields, used when a caller omits columns at inference
RAW_DEFAULTS = {
    "transaction_id": "TXN9990000001",
    "customer_id": "CUST99000000",
    "transaction_date": "2024-01-01",
    "transaction_time": "12:00:00",
    "hour_of_day": 12,
    "is_weekend": 0,
    "is_night_transaction": 0,
    "country": "USA",
    "city": "New York",
    "merchant_category": "Grocery",
    "payment_method": "Debit Card",
    "device_type": "Mobile",
    "customer_age": 35,
    "credit_score": 600,
    "account_age_years": 5.0,
    "account_balance": 1000.0,
    "transaction_amount": 100.0,
    "num_prev_transactions": 10,
    "transaction_freq_monthly": 5,
    "distance_from_home_km": 0.0,
    "time_since_last_txn_hrs": 24.0,
    "is_international": 0,
    "failed_attempts": 0,
    "pin_changed_recently": 0,
    "is_fraud": 0,
    "fraud_type": "",
}

# The exact column order of the original bank_fraud.csv, used for headerless CSV inference
RAW_COLUMNS = [
    'transaction_id',
    'customer_id',
    'transaction_date',
    'transaction_time',
    'hour_of_day',
    'is_weekend',
    'is_night_transaction',
    'country',
    'city',
    'merchant_category',
    'payment_method',
    'device_type',
    'customer_age',
    'credit_score',
    'account_age_years',
    'account_balance',
    'transaction_amount',
    'num_prev_transactions',
    'transaction_freq_monthly',
    'distance_from_home_km',
    'time_since_last_txn_hrs',
    'is_international',
    'failed_attempts',
    'pin_changed_recently',
    'is_fraud',
    'fraud_type'
]


def _get_series(df, col, default):
    """Return a column as a Series, or a constant Series if the column is missing."""
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index)


def _to_int(series):
    return pd.to_numeric(series, errors="coerce").fillna(0).astype(int)


def _to_float(series):
    return pd.to_numeric(series, errors="coerce").fillna(0.0)


def engineer_features(df, sort=True, split_year=2022):
    df = df.copy()

    # Build a single datetime column from date and time strings
    if "transaction_datetime" not in df.columns:
        date_series = _get_series(df, "transaction_date", RAW_DEFAULTS["transaction_date"]).astype(str)
        time_series = _get_series(df, "transaction_time", RAW_DEFAULTS["transaction_time"]).astype(str)
        df["transaction_datetime"] = pd.to_datetime(date_series + " " + time_series, errors="coerce")

    # Sorting is only useful during training preprocessing. During real-time
    # inference it would reorder batch requests and misalign predictions with
    # the original input order, so prepare_transactions() calls with sort=False
    if sort:
        df = df.sort_values("transaction_datetime").reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    # Extract year BEFORE dropping columns or returning
    df["year"] = df["transaction_datetime"].dt.year

    # Numeric Feature Engineering
    transaction_amount = _to_float(_get_series(df, "transaction_amount", RAW_DEFAULTS["transaction_amount"]))
    account_balance = _to_float(_get_series(df, "account_balance", RAW_DEFAULTS["account_balance"]))
    distance_from_home_km = _to_float(_get_series(df, "distance_from_home_km", RAW_DEFAULTS["distance_from_home_km"]))
    time_since_last_txn_hrs = _to_float(_get_series(df, "time_since_last_txn_hrs", RAW_DEFAULTS["time_since_last_txn_hrs"]))
    transaction_freq_monthly = _to_float(_get_series(df, "transaction_freq_monthly", RAW_DEFAULTS["transaction_freq_monthly"]))
    account_age_years = _to_float(_get_series(df, "account_age_years", RAW_DEFAULTS["account_age_years"]))
    num_prev_transactions = _to_float(_get_series(df, "num_prev_transactions", RAW_DEFAULTS["num_prev_transactions"]))

    df["log_transaction_amount"] = np.log1p(transaction_amount)
    df["log_account_balance"] = np.log1p(account_balance)
    df["amount_to_balance_ratio"] = transaction_amount / (account_balance + 1.0)

    # Behavioural flags
    df["is_night_transaction"] = _to_int(_get_series(df, "is_night_transaction", RAW_DEFAULTS["is_night_transaction"]))
    df["is_international"] = _to_int(_get_series(df, "is_international", RAW_DEFAULTS["is_international"]))
    df["failed_attempts"] = _to_int(_get_series(df, "failed_attempts", RAW_DEFAULTS["failed_attempts"]))
    df["pin_changed_recently"] = _to_int(_get_series(df, "pin_changed_recently", RAW_DEFAULTS["pin_changed_recently"]))
    df["is_weekend"] = _to_int(_get_series(df, "is_weekend", RAW_DEFAULTS["is_weekend"]))

    df["risk_score"] = (
        df["is_night_transaction"].astype(int)
        + df["is_international"].astype(int)
        + (df["failed_attempts"] > 0).astype(int)
        + df["pin_changed_recently"].astype(int)
    )

    customer_age = _to_float(_get_series(df, "customer_age", RAW_DEFAULTS["customer_age"]))
    df["customer_age_group"] = pd.cut(
        customer_age,
        bins=[0, 25, 40, 60, 100],
        labels=["18-25", "26-40", "41-60", "60+"],
    ).astype(str)

    hour_of_day = _to_float(_get_series(df, "hour_of_day", RAW_DEFAULTS["hour_of_day"]))
    df["hour_bin"] = pd.cut(
        hour_of_day,
        bins=[-1, 5, 11, 17, 23],
        labels=["night_0_5", "morning_6_11", "afternoon_12_17", "evening_18_23"],
    ).astype(str)

    df["failed_x_transaction"] = df["failed_attempts"] * df["is_international"]
    df["night_x_international"] = df["is_night_transaction"] * df["is_international"]
    df["pin_x_failed"] = df["pin_changed_recently"] * df["failed_attempts"]
    df["night_x_pin"] = df["is_night_transaction"] * df["pin_changed_recently"]
    df["amount_x_distance"] = df["log_transaction_amount"] * df["distance_from_home_km"]
    df["amount_x_failed"] = df["log_transaction_amount"] * df["failed_attempts"]
    df["risk_x_amount"] = df["risk_score"] * df["log_transaction_amount"]
    df["risk_x_distance"] = df["risk_score"] * df["distance_from_home_km"]

    train_period = df["transaction_datetime"].dt.year <= split_year
    if train_period.any():
        amount_p90 = df.loc[train_period, "transaction_amount"].quantile(0.90) if "transaction_amount" in df.columns else transaction_amount[train_period].quantile(0.90)
        distance_p90 = distance_from_home_km[train_period].quantile(0.90)
        time_p10 = time_since_last_txn_hrs[train_period].quantile(0.10)
    else:
        amount_p90 = transaction_amount.quantile(0.90)
        distance_p90 = distance_from_home_km.quantile(0.90)
        time_p10 = time_since_last_txn_hrs.quantile(0.10)

    df["high_amount"] = (transaction_amount > amount_p90).astype(int)
    df["far_from_home"] = (distance_from_home_km > distance_p90).astype(int)
    df["rapid_txn"] = (time_since_last_txn_hrs < time_p10).astype(int)

    has_customer_id = "customer_id" in df.columns
    if has_customer_id and train_period.any():
        customer_stats = df[train_period].groupby("customer_id").agg(
            customer_mean_amount=("transaction_amount", "mean"),
            customer_std_amount=("transaction_amount", "std"),
            customer_mean_distance=("distance_from_home_km", "mean"),
            customer_std_distance=("distance_from_home_km", "std"),
            customer_txn_count=("transaction_amount", "count"),
            customer_mean_balance=("account_balance", "mean"),
        ).reset_index()

        customer_stats["customer_std_amount"] = customer_stats["customer_std_amount"].fillna(0)
        customer_stats["customer_std_distance"] = customer_stats["customer_std_distance"].fillna(0)

        # customer behaviour features (use training period only to avoid leakage)
        df = df.merge(customer_stats, on="customer_id", how="left")

        # Global baseline imputation values
        global_mean_amount = transaction_amount[train_period].mean() if train_period.any() else transaction_amount.mean()
        global_std_amount = transaction_amount[train_period].std() if train_period.any() else transaction_amount.std()
        global_mean_distance = distance_from_home_km[train_period].mean() if train_period.any() else distance_from_home_km.mean()
        global_std_distance = distance_from_home_km[train_period].std() if train_period.any() else distance_from_home_km.std() 
        global_mean_balance = account_balance[train_period].mean() if train_period.any() else account_balance.mean()

        # Fill NaNs (unseen customers)
        df["customer_mean_amount"] = df["customer_mean_amount"].fillna(global_mean_amount)
        df["customer_std_amount"] = df["customer_std_amount"].fillna(global_std_amount)
        df["customer_mean_distance"] = df["customer_mean_distance"].fillna(global_mean_distance)
        df["customer_std_distance"] = df["customer_std_distance"].fillna(global_std_distance)
        df["customer_txn_count"] = df["customer_txn_count"].fillna(1)
        df["customer_mean_balance"] = df["customer_mean_balance"].fillna(global_mean_balance)
    else:
        df["customer_mean_amount"] = transaction_amount.mean()
        df["customer_std_amount"] = transaction_amount.std() if len(df) > 1 else 0
        df["customer_mean_distance"] = distance_from_home_km.mean()
        df["customer_std_distance"] = distance_from_home_km.std() if len(df) > 1 else 0
        df["customer_txn_count"] = 1.0
        df["customer_mean_balance"] = account_balance.mean()

    # Z-scores and deviations
    df["amount_zscore"] = (
        (transaction_amount - df["customer_mean_amount"])
        / (df["customer_std_amount"] + 1e-6)
    )
    df["distance_zscore"] = (
        (distance_from_home_km - df["customer_mean_distance"])
        / (df["customer_std_distance"] + 1e-6)
    )
    df["balance_deviation"] = (
        (account_balance - df["customer_mean_balance"])
        / (df["customer_mean_balance"] + 1e-6)
    )

    # Clean up non-feature metadata columns
    cols_to_drop = [c for c in DROP_COLS if c in df.columns]
    cols_to_drop.append("transaction_datetime")
    df = df.drop(columns=cols_to_drop, errors="ignore")

    return df


def prepare_transactions(data):
    """
    Accept a dict, list of dicts, or DataFrame and return a DataFrame
    with feature columns matched to training outputs.
    """
    if isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        if isinstance(data, dict):
            records = [data]
        else:
            records = list(data)

        if not records:
            return pd.DataFrame(columns=FEATURE_COLUMNS)

        filled = []
        for raw in records:
            record = RAW_DEFAULTS.copy()
            for k, v in raw.items():
                if v is not None and not (isinstance(v, float) and pd.isna(v)):
                    record[k] = v
            filled.append(record)
        df = pd.DataFrame(filled)

    df = engineer_features(df, sort=False)
    return df[FEATURE_COLUMNS]


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--split-year", type=int, default=2022)
    parser.add_argument("--input-dir", type=str, default=os.environ.get("SM_INPUT_DIR", "/opt/ml/processing/input"))
    parser.add_argument("--output-dir", type=str, default=os.environ.get("SM_OUTPUT_DIR", "/opt/ml/processing/output"))
    parser.add_argument("--test-size", type=float, default=0.20)
    parser.add_argument("--random-state", type=int, default=42)
    return parser.parse_args()


def main():
    args = parse_args()

    csv_files = glob.glob(os.path.join(args.input_dir, '**', '*.csv'), recursive=True)
    if not csv_files:
        raise FileNotFoundError(f'No CSV found in {args.input_dir}')
    input_path = csv_files[0]
    os.makedirs(args.output_dir, exist_ok=True)

    logger.info(f"Loading raw data from {input_path}")
    try:
        df = pd.read_csv(input_path)
        if set(df.columns) != set(RAW_COLUMNS):
            df = pd.read_csv(input_path, names=RAW_COLUMNS, header=None)
    except Exception:
        df = pd.read_csv(input_path, names=RAW_COLUMNS, header=None)

    df = engineer_features(df)

    full_df = df[FEATURE_COLUMNS + [TARGET_COLUMN]]
    train_df, test_df = train_test_split(
        full_df, test_size=args.test_size, stratify=full_df[TARGET_COLUMN], random_state=args.random_state,
    )

    train_dir = os.path.join(args.output_dir, "train")
    test_dir = os.path.join(args.output_dir, "test")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    
    # Save features and labels
    train_df[FEATURE_COLUMNS].to_csv(os.path.join(train_dir, "train_features.csv"), index=False)
    train_df[TARGET_COLUMN].to_frame().to_csv(os.path.join(train_dir, "train_labels.csv"), index=False)

    test_df[FEATURE_COLUMNS].to_csv(os.path.join(test_dir, "test_features.csv"), index=False)
    test_df[TARGET_COLUMN].to_frame().to_csv(os.path.join(test_dir, "test_labels.csv"), index=False)

    logger.info(f"Preprocessing complete. Train: {len(train_df)} rows | Test: {len(test_df)} rows")


if __name__ == "__main__":
    main()


Overwriting triggered_pipeline_src/preprocess.py


**Insight — `train.py` here implements the same XGBoost + `SMOTE` architecture as Notebook 03**. The enhanced pipeline is an upgrade of the *process* (adding structured evaluation and a retraining trigger design).

In [35]:
%%writefile triggered_pipeline_src/train.py
import argparse
import json
import logging
import os
import subprocess
import sys

# Dynamic dependency resolution for SageMaker pre-built containers
try:
    import imblearn
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "imbalanced-learn"])
    import imblearn

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier

from preprocess import CATEGORICAL_FEATURES, FEATURE_COLUMNS, NUMERIC_FEATURES

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

TARGET_COLUMN = "is_fraud"
MODEL_FILENAME = "model.joblib"
CONFIG_FILENAME = "model_config.json"


def parse_args():
    parser = argparse.ArgumentParser()

    # Hyperparameters
    parser.add_argument("--n-estimators", "--n_estimators", type=int, default=500, dest="n_estimators")
    parser.add_argument("--max-depth", "--max_depth", type=int, default=8, dest="max_depth")
    parser.add_argument("--learning-rate", "--learning_rate", type=float, default=0.05, dest="learning_rate")
    parser.add_argument("--smote-sampling-strategy", "--smote_sampling_strategy", type=float, default=0.5, dest="smote_sampling_strategy")
    parser.add_argument("--random-state", type=int, default=42)
    parser.add_argument("--n-jobs", type=int, default=-1)

    # Metadata
    parser.add_argument("--team-id", type=str, default=os.environ.get("TEAM_ID", "unknown-team"))
    parser.add_argument("--student-id", type=str, default=os.environ.get("STUDENT_ID", "s000"))
    parser.add_argument("--semester", type=str, default=os.environ.get("SEMESTER", "26S1"))
    parser.add_argument("--run-name", type=str, default="sagemaker_pipeline_run")

    # SageMaker channels / dirs
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST", "/opt/ml/input/data/test"))
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model"))

    args, _ = parser.parse_known_args()
    return args


def _best_f1_threshold(y_true, y_prob):
    """Return the probability threshold that maximises the F1 score on y_true."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_idx = int(np.argmax(f1))
    return float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.5


def _load_data(channel_dir, split_name):
    """Safely loads feature/label files or splits a single CSV if features/labels aren't separate."""
    features_path = os.path.join(channel_dir, f"{split_name}_features.csv")
    labels_path = os.path.join(channel_dir, f"{split_name}_labels.csv")
    combined_path = os.path.join(channel_dir, f"{split_name}.csv")

    if os.path.exists(features_path) and os.path.exists(labels_path):
        X = pd.read_csv(features_path)
        y = pd.read_csv(labels_path).squeeze("columns")
    elif os.path.exists(combined_path):
        df = pd.read_csv(combined_path)
        y = df[TARGET_COLUMN]
        X = df.drop(columns=[TARGET_COLUMN], errors="ignore")
    else:
        raise FileNotFoundError(f"Could not find valid dataset files in channel directory: {channel_dir}")

    X = X.drop(columns=[TARGET_COLUMN], errors="ignore")
    return X, y


def main():
    args = parse_args()
    os.makedirs(args.model_dir, exist_ok=True)

    X_train, y_train = _load_data(args.train, "train")
    X_test, y_test = _load_data(args.test, "test")

    if len(pd.Series(y_train).unique()) < 2:
        raise ValueError("Training labels contain fewer than two classes.")

    X_fit, X_val, y_fit, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        stratify=y_train,
        random_state=args.random_state,
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), [c for c in NUMERIC_FEATURES if c in X_fit.columns]),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32),
                [c for c in CATEGORICAL_FEATURES if c in X_fit.columns],
            ),
        ],
        remainder="passthrough",
    )

    # Compute actual minority balance safely
    class_counts = pd.Series(y_fit).value_counts()
    minority_count = class_counts.min()
    majority_count = class_counts.max()
    current_ratio = minority_count / majority_count

    # Validate SMOTE pre-conditions
    # Target ratio must strictly exceed current ratio and minority count must be > k_neighbors (default 5)
    target_smote_ratio = args.smote_sampling_strategy
    if current_ratio < target_smote_ratio and minority_count > 5:
        k_neighbors = min(5, minority_count - 1)
        smote_step = SMOTE(
            sampling_strategy=target_smote_ratio,
            k_neighbors=k_neighbors,
            random_state=args.random_state,
        )
        logger.info(f"Applying SMOTE with target ratio {target_smote_ratio:.2f} (current: {current_ratio:.2f})")
    else:
        smote_step = "passthrough"
        logger.info(
            f"Bypassing SMOTE: current ratio ({current_ratio:.2f}) meets/exceeds target ({target_smote_ratio:.2f}) "
            f"or insufficient minority samples ({minority_count})."
        )

    model = ImbPipeline([
        ("prep", preprocessor),
        ("smote", smote_step),
        ("clf", XGBClassifier(
            n_estimators=args.n_estimators,
            max_depth=args.max_depth,
            learning_rate=args.learning_rate,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=5,
            gamma=0.1,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=args.random_state,
            eval_metric="aucpr",
            tree_method="hist",
            n_jobs=args.n_jobs,
        )),
    ])
    model.fit(X_fit, y_fit)

    val_probs = model.predict_proba(X_val)[:, 1]
    best_threshold = _best_f1_threshold(y_val, val_probs)
    logger.info(f"Best F1 threshold on validation: {best_threshold:.4f}")

    all_metrics = {}
    for split, X, y in [("train", X_fit, y_fit), ("test", X_test, y_test)]:
        probabilities = model.predict_proba(X)[:, 1]
        predictions = (probabilities >= best_threshold).astype(int)

        all_metrics.update(
            {
                f"{split}_accuracy": round(accuracy_score(y, predictions), 4),
                f"{split}_f1": round(f1_score(y, predictions, zero_division=0), 4),
                f"{split}_precision": round(precision_score(y, predictions, zero_division=0), 4),
                f"{split}_recall": round(recall_score(y, predictions, zero_division=0), 4),
            }
        )

        if len(pd.Series(y).unique()) >= 2:
            all_metrics[f"{split}_roc_auc"] = round(roc_auc_score(y, probabilities), 4)
            all_metrics[f"{split}_pr_auc"] = round(average_precision_score(y, probabilities), 4)
        else:
            all_metrics[f"{split}_roc_auc"] = None
            all_metrics[f"{split}_pr_auc"] = None
            logger.warning(f"{split} split has only one class; AUC-ROC unavailable.")

    logger.info("=== Metrics ===")
    for metric_name, metric_value in sorted(all_metrics.items()):
        logger.info(f"  {metric_name:<20}: {metric_value}")

    model_path = os.path.join(args.model_dir, MODEL_FILENAME)
    joblib.dump(model, model_path)
    logger.info(f"Model saved: {model_path}")

    config = {
        "threshold": round(best_threshold, 4),
        "feature_columns": FEATURE_COLUMNS,
        "target_column": TARGET_COLUMN,
        "label_map": {"0": "Non-Fraud", "1": "Fraud"},
        "model_type": "SMOTE+XGBoost",
    }

    config_path = os.path.join(args.model_dir, CONFIG_FILENAME)
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)
    logger.info(f"Model config saved: {config_path}")

    if all_metrics["test_roc_auc"] is None:
        raise ValueError("Test AUC-ROC is unavailable; cannot evaluate the quality gate.")

    print(f"test_pr_auc: {all_metrics['test_pr_auc']}")
    print(f"Test AUC-ROC: {all_metrics['test_roc_auc']}")
    print(f"test_accuracy: {all_metrics['test_accuracy']}")
    print(f"test_f1: {all_metrics['test_f1']}")
    print(f"test_recall: {all_metrics['test_recall']}")
    print(f"test_precision: {all_metrics['test_precision']}")
    print(f"best_threshold: {best_threshold:.4f}")


if __name__ == "__main__":
    main()

Overwriting triggered_pipeline_src/train.py


**Finding — a dedicated `EvaluateModel` step reloads the trained model, scores the test set directly, and writes a structured `evaluation.json` exposed via a SageMaker PropertyFile, rather than relying on the training job's printed log text.** This directly resolves the fragility identified in Notebook 03, where the quality-gate condition depended on the exact text format `train.py` prints — a coupling that could silently break if that format ever changed.

In [60]:
%%writefile triggered_pipeline_src/evaluate.py
import os
import sys
import glob
import json
import tarfile
import argparse
import subprocess

# Ensure required libraries are installed before unpickling the model
required_packages = ["imbalanced-learn", "xgboost"]
for package in required_packages:
    try:
        import_name = "imblearn" if package == "imbalanced-learn" else package
        __import__(import_name)
    except ImportError:
        print(f"Installing missing package: {package}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import joblib
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
)


def find_file(folder: str, filename: str) -> str:
    expected_path = os.path.join(folder, filename)

    if os.path.exists(expected_path):
        return expected_path

    matches = glob.glob(
        os.path.join(folder, "**", filename),
        recursive=True
    )

    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not find {filename} under {folder}. "
        f"Found: {matches}. "
        f"All files: {glob.glob(os.path.join(folder, '**', '*'), recursive=True)}"
    )


def extract_model_tar_if_needed(model_dir: str):
    """
    SageMaker TrainingStep passes model.tar.gz into the evaluation step.
    Extracts archive so model.joblib is available.
    """
    print("Listing model directory before extraction:")
    for root, dirs, files in os.walk(model_dir):
        print(root, files)

    tar_files = glob.glob(
        os.path.join(model_dir, "**", "model.tar.gz"),
        recursive=True
    )

    if not tar_files:
        print("No model.tar.gz found. Looking for model.joblib directly...")
        return

    tar_path = tar_files[0]
    print("Found model.tar.gz:", tar_path)
    print("Extracting model.tar.gz to:", model_dir)

    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=model_dir)

    print("Listing model directory after extraction:")
    for root, dirs, files in os.walk(model_dir):
        print(root, files)


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--model-dir",
        type=str,
        default="/opt/ml/processing/model"
    )

    parser.add_argument(
        "--test",
        type=str,
        default="/opt/ml/processing/test"
    )

    parser.add_argument(
        "--output-dir",
        type=str,
        default="/opt/ml/processing/evaluation"
    )

    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)

    print("=== Evaluation Environment ===")
    print("Model dir:", args.model_dir)
    print("Test dir:", args.test)
    print("Output dir:", args.output_dir)

    extract_model_tar_if_needed(args.model_dir)

    model_file = find_file(args.model_dir, "model.joblib")
    config_file = find_file(args.model_dir, "model_config.json")
    test_features_file = find_file(args.test, "test_features.csv")
    test_labels_file = find_file(args.test, "test_labels.csv")

    print("Model file:", model_file)
    print("Config file:", config_file)
    print("Test features:", test_features_file)
    print("Test labels:", test_labels_file)

    model = joblib.load(model_file)
    with open(config_file, "r", encoding='utf-8') as f:
        config = json.load(f)

    threshold = config.get('threshold', 0.5)
    target_column = config.get('target_column', 'is_fraud')

    X_test = pd.read_csv(test_features_file)
    y_test = pd.read_csv(test_labels_file).squeeze("columns")

    # Drop target column if present in feature matrix
    X_test = X_test.drop(columns=[target_column], errors='ignore')

    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= threshold).astype(int)

    metrics = {
        "accuracy": round(accuracy_score(y_test, predictions), 4),
        "f1": round(f1_score(y_test, predictions, zero_division=0), 4),
        "precision": round(precision_score(y_test, predictions, zero_division=0), 4),
        "recall": round(recall_score(y_test, predictions, zero_division=0), 4),
    }

    if len(pd.Series(y_test).unique()) >= 2:
        metrics["auc_roc"] = round(roc_auc_score(y_test, probabilities), 4)
    else:
        metrics["auc_roc"] = 0.0

    evaluation_report = {
        "classification_metrics": {
            "auc_roc": {
                "value": metrics["auc_roc"]
            },
            "accuracy": {
                "value": metrics["accuracy"]
            },
            "f1": {
                "value": metrics["f1"]
            },
            "precision": {
                "value": metrics["precision"]
            },
            "recall": {
                "value": metrics["recall"]
            }
        }
    }

    output_path = os.path.join(args.output_dir, "evaluation.json")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(evaluation_report, f, indent=2)

    print("Saved evaluation report:", output_path)
    print(json.dumps(evaluation_report, indent=2))


if __name__ == "__main__":
    main()

Overwriting triggered_pipeline_src/evaluate.py


**Insight — `inference.py` is kept identical to Notebook 03's version**, so an endpoint deployed from either the original or the enhanced pipeline behaves identically from a client's perspective, including the Gradio demo in Notebook 04.

In [61]:
%%writefile triggered_pipeline_src/inference.py
import json
import os

import joblib
from preprocess import FEATURE_COLUMNS, prepare_transactions

MODEL_FILENAME = "model.joblib"
CONFIG_FILENAME = "model_config.json"

# Holds the tuned decision threshold loaded by model_fn so that
# predict_fn can apply the same operating point used during training.
# Falls back to 0.5 if no config is present (e.g. when testing an
# older model artifact).
MODEL_CONFIG = {}


def model_fn(model_dir):
    """Load the fitted sklearn Pipeline and its companion config file."""
    global MODEL_CONFIG
    model_path = os.path.join(model_dir, MODEL_FILENAME)
    config_path = os.path.join(model_dir, CONFIG_FILENAME)
    
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f"Expected model artifact at {model_path}, found: {os.listdir(model_dir)}")

    model = joblib.load(model_path)

    MODEL_CONFIG = {}
    if os.path.isfile(config_path):
        with open(config_path, "r", encoding="utf-8") as f:
            MODEL_CONFIG = json.load(f)
        print(f"Loaded model config: threshold={MODEL_CONFIG.get('threshold', 0.5)}")
    else:
        print("No model_config.json found, using default threshold 0.5")
    return model


def input_fn(body, content_type="application/json"):
    """Parse the request body and return a DataFrame with model-ready columns."""
    # Strip optional attributes like charset=utf-8
    content_type_clean = content_type.split(";")[0].strip().lower()

    if content_type_clean != "application/json":
        raise ValueError(f"Unsupported content type: {content_type}")

    payload = json.loads(body)
    records = payload if isinstance(payload, list) else [payload]

    # prepare_transactions already returns df[FEATURE_COLUMNS]
    return prepare_transactions(records)


def predict_fn(data, model):
    """Return both predicted class and fraud probability."""
    threshold = MODEL_CONFIG.get("threshold", 0.5)
    probs = model.predict_proba(data)[:, 1]
    preds = (probs >= threshold).astype(int)

    return preds, probs


def output_fn(prediction, accept="application/json"):
    """Serialize predictions as JSON."""
    label_map = MODEL_CONFIG.get("label_map", {"0": "Non-Fraud", "1": "Fraud"})
    preds, probs = prediction
    response = [
        {
            "prediction": int(p),
            "label": label_map.get(str(int(p)), "Fraud" if int(p) == 1 else "Non-Fraud"),
            "probability": round(float(b), 4),
        }
        for p, b in zip(preds, probs)
    ]
    return json.dumps(response), "application/json"

Overwriting triggered_pipeline_src/inference.py


In [62]:
# Ensure the Model Package Group exists
try:
    sm.create_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
        ModelPackageGroupDescription=(
            f"Model package group for {TEAM_ID} S3-triggered Bank Fraud Detection pipeline"
        )
    )
    print("Created model package group:", MODEL_PACKAGE_GROUP_NAME)

except botocore.exceptions.ClientError as e:
    error_code = e.response.get("Error", {}).get("Code", "")
    error_message = e.response.get("Error", {}).get("Message", "")

    if "already exists" in error_message.lower() or error_code == "ValidationException":
        print("Model package group already exists:", MODEL_PACKAGE_GROUP_NAME)
    else:
        raise

Model package group already exists: team09-BankFraudDetection-Triggered


In [63]:
# Build the SageMaker Pipeline

# Processor is used by both preprocessing and evaluation steps.
# Important: SKLearnProcessor does not support code_location.
# The safe S3 upload location comes from PipelineSession(default_bucket_prefix=TEAM_PREFIX).
processor = SKLearnProcessor(
    framework_version="1.4-2",
    role=ROLE_ARN,
    instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1,
    base_job_name=f"iti113-{TEAM_ID}-process",
    sagemaker_session=pipeline_session,
)


# Estimator is used by the TrainingStep.
# output_path keeps model artifacts under the team prefix.
estimator = SKLearn(
    entry_point="train.py",
    source_dir=str(SRC_DIR),
    framework_version="1.4-2",
    py_version="py3",
    role=ROLE_ARN,
    instance_type=TRAINING_INSTANCE_TYPE,
    instance_count=1,
    base_job_name=f"iti113-{TEAM_ID}-train",
    sagemaker_session=pipeline_session,
    code_location=TRAINING_CODE_LOCATION,
    output_path=TRAINING_OUTPUT_PREFIX,
    hyperparameters={
        "n-estimators": n_estimators_param,
        "max-depth": max_depth_param,
        "random-state": 42,
        "team-id": TEAM_ID,
        "student-id": STUDENT_ID,
        "semester": SEMESTER,
        "run-name": "s3_triggered_pipeline_run",
    },
)


# Step 1: Preprocess the CSV file passed in through the InputDataUrl parameter.
step_process = ProcessingStep(
    name="PreprocessData",
    processor=processor,
    inputs=[
        ProcessingInput(
            source=input_data_url_param,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/output/train"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/output/test"
        ),
    ],
    code=str(SRC_DIR / "preprocess.py"),
    job_arguments=[
        "--test-size", "0.2",
        "--random-state", "42",
    ],
)


# Step 2: Train model using outputs from preprocessing.
step_train = TrainingStep(
    name="TrainModel",
    estimator=estimator,
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs[
                "train"
            ].S3Output.S3Uri,
            content_type="text/csv",
        ),
        "test": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs[
                "test"
            ].S3Output.S3Uri,
            content_type="text/csv",
        ),
    },
)


# Step 3: Evaluate model.
# The TrainingStep model artifact arrives as model.tar.gz, so evaluate.py extracts it first.
evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

step_eval = ProcessingStep(
    name="EvaluateModel",
    processor=processor,
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model"
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs[
                "test"
            ].S3Output.S3Uri,
            destination="/opt/ml/processing/test"
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation"
        )
    ],
    code=str(SRC_DIR / "evaluate.py"),
    property_files=[evaluation_report],
)


# Step 4: Define model metrics used in the Model Registry.
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0][
                "S3Output"
            ]["S3Uri"]
        ),
        content_type="application/json",
    )
)

# Step 5: Register model if quality gate passes.
model = SKLearnModel(
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=ROLE_ARN,
    entry_point="inference.py",
    source_dir=str(SRC_DIR),
    framework_version="1.4-2",
    py_version="py3",
    sagemaker_session=pipeline_session,
    env={
        "HOME": "/tmp",
        "PYTHONUSERBASE": "/tmp/.local",
        "PYTHONNOUSERSITE": "0",
    },
)

step_register = ModelStep(
    name="RegisterModel",
    step_args=model.register(
        content_types=["application/json"],
        response_types=["application/json"],
        inference_instances=["ml.m5.large"],
        transform_instances=["ml.m5.large"],
        model_package_group_name=MODEL_PACKAGE_GROUP_NAME,
        approval_status="PendingManualApproval",
        model_metrics=model_metrics,
    )
)

# Step 6: Quality gate.
# If AUC is below QualityGateAUC, the pipeline still succeeds but registration is skipped.
cond_auc = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="classification_metrics.auc_roc.value",
    ),
    right=quality_gate_auc_param,
)

step_cond = ConditionStep(
    name="CheckAUCQualityGate",
    conditions=[cond_auc],
    if_steps=[step_register],
    else_steps=[],
)


# Create the pipeline object.
triggered_pipeline = Pipeline(
    name=TRIGGERED_PIPELINE_NAME,
    parameters=[
        input_data_url_param,
        n_estimators_param,
        max_depth_param,
        quality_gate_auc_param,
    ],
    steps=[
        step_process,
        step_train,
        step_eval,
        step_cond,
    ],
    sagemaker_session=pipeline_session,
)

print("Triggered pipeline object created:")
print(TRIGGERED_PIPELINE_NAME)


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


Triggered pipeline object created:
iti113-team09-bank-fraud-detection-triggered


**Insight — the pipeline exposes an `InputDataUrl` parameter, so any freshly uploaded S3 CSV can be scored/retrained against, not only the original static dataset** — this parameterisation is what makes the pipeline genuinely re-triggerable rather than a one-shot script.

In [64]:
# Validate that the generated pipeline definition is valid JSON.
definition = triggered_pipeline.definition()
definition_json = json.loads(definition)

print("Pipeline definition is valid JSON.")

print("\nPipeline parameters:")
print(json.dumps(definition_json.get("Parameters", []), indent=2))

# Create or update the pipeline in SageMaker.
upsert_response = triggered_pipeline.upsert(
    role_arn=ROLE_ARN
)

print("\nPipeline upsert completed.")
print(json.dumps(upsert_response, indent=2, default=str))

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pipeline definition is valid JSON.

Pipeline parameters:
[
  {
    "Name": "InputDataUrl",
    "Type": "String",
    "DefaultValue": "s3://nyp-26s1-iti113/iti113/team09/data/bank-fraud-detection/raw/bank_fraud.csv"
  },
  {
    "Name": "NEstimators",
    "Type": "Integer",
    "DefaultValue": 500
  },
  {
    "Name": "MaxDepth",
    "Type": "Integer",
    "DefaultValue": 8
  },
  {
    "Name": "QualityGateAUC",
    "Type": "Float",
    "DefaultValue": 0.7
  }
]



Pipeline upsert completed.
{
  "PipelineArn": "arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team09-bank-fraud-detection-triggered",
  "PipelineVersionId": 6,
  "ResponseMetadata": {
    "RequestId": "4c5f2db4-0ad0-43d1-b1c0-0e2beb2be10f",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "4c5f2db4-0ad0-43d1-b1c0-0e2beb2be10f",
      "strict-transport-security": "max-age=47304000; includeSubDomains",
      "x-frame-options": "DENY",
      "content-security-policy": "frame-ancestors 'none'",
      "cache-control": "no-cache, no-store, must-revalidate",
      "x-content-type-options": "nosniff",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "139",
      "date": "Sun, 23 Aug 2026 10:45:03 GMT"
    },
    "RetryAttempts": 0
  }
}


In [65]:
RAW_COLUMNS = [
    "transaction_id",
    "customer_id",
    "transaction_date",
    "transaction_time",
    "hour_of_day",
    "is_weekend",
    "is_night_transaction",
    "country",
    "city",
    "merchant_category",
    "payment_method",
    "device_type",
    "customer_age",
    "credit_score",
    "account_age_years",
    "account_balance",
    "transaction_amount",
    "num_prev_transactions",
    "transaction_freq_monthly",
    "distance_from_home_km",
    "time_since_last_txn_hrs",
    "is_international",
    "failed_attempts",
    "pin_changed_recently",
    "is_fraud",
    "fraud_type",
]

np.random.seed(42)
n = 1000

records = []
for i in range(n):
    hour = int(np.random.choice(range(24)))
    is_night = 1 if hour < 6 else 0
    is_weekend = int(np.random.choice([0, 1]))
    amount = max(10.0, round(np.random.exponential(300), 2))
    balance = max(100.0, round(np.random.exponential(2000), 2))
    failed = int(
        np.random.choice([0, 1, 2, 3, 4, 5], p=[0.60, 0.15, 0.10, 0.08, 0.05, 0.02])
    )
    international = int(np.random.choice([0, 1], p=[0.85, 0.15]))
    pin_changed = int(np.random.choice([0, 1], p=[0.90, 0.10]))
    distance = round(np.random.exponential(15), 2)
    credit = int(np.random.randint(300, 850))
    acc_age = round(np.random.uniform(0.5, 20), 1)

    risk = (
        2.0 * min(failed, 5)
        + 1.2 * is_night
        + 1.2 * international
        + 1.0 * pin_changed
        + 0.8 * (1 if amount > 300 else 0)
        + 0.7 * (1 if distance > 40 else 0)
        + 1.0 * is_night * international
        + 0.8 * (1 if failed > 0 else 0) * international
        + 0.5 * (1 if credit < 450 else 0)
        + np.random.normal(0, 0.3)
    )

    is_fraud = 1 if risk > 4.0 else 0

    records.append(
        {
            "transaction_id": f"TXN{100000 + i:08d}",
            "customer_id": f"CUST{900000 + i:06d}",
            "transaction_date": "2024-06-04",
            "transaction_time": f"{hour:02d}:00:00",
            "hour_of_day": hour,
            "is_weekend": is_weekend,
            "is_night_transaction": is_night,
            "country": np.random.choice(
                [
                    "Australia",
                    "Brazil",
                    "Canada",
                    "France",
                    "Germany",
                    "India",
                    "Japan",
                    "Mexico",
                    "UK",
                    "USA",
                ]
            ),
            "city": np.random.choice(
                [
                    "Berlin",
                    "Delhi",
                    "Guadalajara",
                    "London",
                    "Los Angeles",
                    "Lyon",
                    "Manchester",
                    "Melbourne",
                    "Mexico City",
                    "Mumbai",
                    "Munich",
                    "New York",
                    "Osaka",
                    "Paris",
                    "Rio",
                    "Sydney",
                    "São Paulo",
                    "Tokyo",
                    "Toronto",
                    "Vancouver",
                ]
            ),
            "merchant_category": np.random.choice(
                [
                    "ATM Withdrawal",
                    "Clothing",
                    "Crypto Exchange",
                    "Education",
                    "Electronics",
                    "Entertainment",
                    "Fuel",
                    "Gaming",
                    "Grocery",
                    "Healthcare",
                    "Jewelry",
                    "Online Shopping",
                    "Restaurant",
                    "Travel",
                    "Utilities",
                ]
            ),
            "payment_method": np.random.choice(
                [
                    "Bank Transfer",
                    "Cheque",
                    "Credit Card",
                    "Crypto",
                    "Debit Card",
                    "Mobile Payment",
                ]
            ),
            "device_type": np.random.choice(
                ["ATM", "Desktop", "Mobile", "POS Terminal", "Tablet"]
            ),
            "customer_age": int(np.random.randint(18, 80)),
            "credit_score": credit,
            "account_age_years": acc_age,
            "account_balance": balance,
            "transaction_amount": amount,
            "num_prev_transactions": int(np.random.randint(0, 200)),
            "transaction_freq_monthly": int(np.random.randint(1, 30)),
            "distance_from_home_km": distance,
            "time_since_last_txn_hrs": round(np.random.exponential(24), 2),
            "is_international": international,
            "failed_attempts": failed,
            "pin_changed_recently": pin_changed,
            "is_fraud": is_fraud,
            "fraud_type": ""
            if is_fraud == 0
            else np.random.choice(
                ["Phishing", "Account Takeover", "Card Cloning", "Identity Theft"]
            ),
        }
    )

raw_df = pd.DataFrame(records, columns=RAW_COLUMNS)

# Downsample the majority class so the dataset is balanced
fraud_df = raw_df[raw_df["is_fraud"] == 1]
non_fraud_df = raw_df[raw_df["is_fraud"] == 0]

n_balanced = min(len(fraud_df), len(non_fraud_df))
if n_balanced == 0:
    raise ValueError("Generated data contains ony one class; cannot balance")

fraud_sample = fraud_df.sample(n=n_balanced, random_state=42)
non_fraud_sample = non_fraud_df.sample(n=n_balanced, random_state=42)

synthetic_df = (
    pd.concat([fraud_sample, non_fraud_sample])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

synthetic_df.to_csv(LOCAL_FRAUD_FILE, index=False)

print("Synthetic fraud CSV saved:", LOCAL_FRAUD_FILE)
print("Shape", synthetic_df.shape)
print("Fraud distribution")
print(synthetic_df["is_fraud"].value_counts())


Synthetic fraud CSV saved: bank_fraud.csv
Shape (576, 26)
Fraud distribution
is_fraud
1    288
0    288
Name: count, dtype: int64


In [66]:
# Verify the local CSV file
print("File exists:", os.path.exists(LOCAL_FRAUD_FILE))
print("File size:", os.path.getsize(LOCAL_FRAUD_FILE), "bytes")

df = pd.read_csv(LOCAL_FRAUD_FILE)

print("Shape:", df.shape)
display(df.head())

if set(df.columns) != set(RAW_COLUMNS):
    raise ValueError("CSV columns do not match the expected CSV format.")

if df['is_fraud'].nunique() < 2:
    raise ValueError("Expected both target classes 0 and 1.")


File exists: True
File size: 87574 bytes
Shape: (576, 26)


,transaction_id,customer_id,transaction_date,transaction_time,hour_of_day,is_weekend,is_night_transaction,country,city,merchant_category,...,transaction_amount,num_prev_transactions,transaction_freq_monthly,distance_from_home_km,time_since_last_txn_hrs,is_international,failed_attempts,pin_changed_recently,is_fraud,fraud_type
0,TXN00100711,CUST900711,2024-06-04,04:00:00,4,0,1,USA,São Paulo,ATM Withdrawal,...,44.28,190,25,8.23,23.39,0,3,0,1,Card Cloning
1,TXN00100217,CUST900217,2024-06-04,22:00:00,22,1,0,Germany,Sydney,Fuel,...,428.67,142,9,18.47,15.16,1,1,0,1,Identity Theft
2,TXN00100491,CUST900491,2024-06-04,01:00:00,1,0,1,Brazil,Vancouver,Education,...,181.14,167,17,7.43,95.16,0,0,0,0,NaN
3,TXN00100529,CUST900529,2024-06-04,01:00:00,1,0,1,UK,Delhi,Restaurant,...,1180.75,83,18,12.96,22.40,0,0,0,0,NaN
4,TXN00100816,CUST900816,2024-06-04,05:00:00,5,1,1,Mexico,Paris,Crypto Exchange,...,244.18,169,13,20.68,13.27,0,0,0,0,NaN


In [67]:
# Manual pipeline test
if not os.path.exists(LOCAL_FRAUD_FILE):
    raise FileNotFoundError(
        f"{LOCAL_FRAUD_FILE} not found. Run the CSV creation cell first."
    )

timestamp = int(time.time())

manual_s3_key = f"{MANUAL_TEST_PREFIX}/" f"fraud_manual_{timestamp}.csv"

s3.upload_file(LOCAL_FRAUD_FILE, BUCKET, manual_s3_key)

manual_s3_uri = f"s3://{BUCKET}/{manual_s3_key}"

print("Uploaded manual test file:")
print(manual_s3_uri)

Uploaded manual test file:
s3://nyp-26s1-iti113/iti113/team09/manual-input/fraud_manual_1787481907.csv


**Finding — the S3-upload-triggered retraining workflow was validated by simulation: a fresh CSV uploaded to a dedicated `manual-input/` S3 prefix, followed by a programmatic `start_pipeline_execution` call, completed all five steps (PreprocessData, TrainModel, EvaluateModel, CheckAUCQualityGate, RegisterModel) to a Succeeded status against a QualityGateAUC of 0.70.** This proves the pipeline correctly handles the kind of call an EventBridge rule/Lambda function would issue automatically — but that rule itself was not deployed as standing infrastructure, so today's trigger is a validated design, not a live, always-on automation.

In [68]:
execution_name = f"manual-triggered-test-{int(time.time())}"

response = sm.start_pipeline_execution(
    PipelineName=TRIGGERED_PIPELINE_NAME,
    PipelineExecutionDisplayName=execution_name,
    PipelineParameters=[
        {"Name": "InputDataUrl", "Value": manual_s3_uri},
        {"Name": "NEstimators", "Value": "150"},
        {"Name": "MaxDepth", "Value": "6"},
        {"Name": "QualityGateAUC", "Value": "0.70"},
    ],
)

TRIGGERED_EXECUTION_ARN = response["PipelineExecutionArn"]

print("Manual pipeline execution started:")
print(TRIGGERED_EXECUTION_ARN)

Manual pipeline execution started:
arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team09-bank-fraud-detection-triggered/execution/xhfheuuda6y0


In [69]:
def print_failed_step_logs(execution_arn):
    client = boto3.client("sagemaker")
    logs_client = boto3.client("logs")
    
    steps = client.list_pipeline_execution_steps(
        PipelineExecutionArn=execution_arn
    )["PipelineExecutionSteps"]
    
    for step in steps:
        if step["StepStatus"] == "Failed":
            step_name = step["StepName"]
            step_type = step.get("StepType", "Unknown")
            print(f"\n--- Fetching Logs for Failed Step: {step_name} (Type: {step_type}) ---")
            
            # Print full step metadata for debugging
            print(f"Full step metadata: {step}")
            
            job_name = None
            log_group_name = None
            
            # Try different metadata structures
            metadata = step.get("Metadata", {})
            
            # Check for TrainingJob
            if "TrainingJob" in metadata:
                job_metadata = metadata["TrainingJob"]
                if "Arn" in job_metadata:
                    job_name = job_metadata["Arn"].split("/")[-1]
                    log_group_name = "/aws/sagemaker/TrainingJobs"
                elif "TrainingJobName" in job_metadata:
                    job_name = job_metadata["TrainingJobName"]
                    log_group_name = "/aws/sagemaker/TrainingJobs"
            
            # Check for ProcessingJob
            elif "ProcessingJob" in metadata:
                job_metadata = metadata["ProcessingJob"]
                if "Arn" in job_metadata:
                    job_name = job_metadata["Arn"].split("/")[-1]
                    log_group_name = "/aws/sagemaker/ProcessingJobs"
                elif "ProcessingJobName" in job_metadata:
                    job_name = job_metadata["ProcessingJobName"]
                    log_group_name = "/aws/sagemaker/ProcessingJobs"
            
            # Check for other job types
            elif "TransformJob" in metadata:
                job_metadata = metadata["TransformJob"]
                if "Arn" in job_metadata:
                    job_name = job_metadata["Arn"].split("/")[-1]
                    log_group_name = "/aws/sagemaker/TransformJobs"
            
            # If we couldn't find metadata, try to extract from step parameters
            if not job_name:
                # Check if the step has parameters that reference a job
                if "Parameters" in step:
                    params = step.get("Parameters", {})
                    # For TrainingJob steps, the job name might be in parameters
                    if "TrainingJobName" in params:
                        job_name = params["TrainingJobName"]
                        log_group_name = "/aws/sagemaker/TrainingJobs"
                    elif "ProcessingJobName" in params:
                        job_name = params["ProcessingJobName"]
                        log_group_name = "/aws/sagemaker/ProcessingJobs"
            
            # Check if we have a failure reason that might contain job info
            if not job_name and "FailureReason" in step:
                failure_reason = step["FailureReason"]
                print(f"Step Failure Reason: {failure_reason}")
                # Some failure reasons contain the job name
                import re
                job_match = re.search(r"arn:aws:sagemaker:[^:]+:[^:]+:training-job/([^\s]+)", failure_reason)
                if job_match:
                    job_name = job_match.group(1)
                    log_group_name = "/aws/sagemaker/TrainingJobs"
                else:
                    # Try processing job pattern
                    job_match = re.search(r"arn:aws:sagemaker:[^:]+:[^:]+:processing-job/([^\s]+)", failure_reason)
                    if job_match:
                        job_name = job_match.group(1)
                        log_group_name = "/aws/sagemaker/ProcessingJobs"
            
            if not job_name or not log_group_name:
                print(f"Could not determine CloudWatch Log Group for step: {step_name}")
                print("This might be because:")
                print("1. The step failed before creating a job")
                print("2. The step is not a Training/Processing/Transform job")
                print("3. Check the SageMaker Console for more details")
                continue
            
            print(f"Attempting to fetch logs for {job_name} from {log_group_name}")

            try:
                job_desc = client.describe_training_job(TrainingJobName=job_name) \
                    if log_group_name.endswith("TrainingJobs") \
                    else client.describe_processing_job(ProcessingJobName=job_name)
                print(f"JobStatus: {job_desc.get('TrainingJobStatus') or job_desc.get('ProcessingJobStatus')}")
                print(f"JobFailureReason: {job_desc.get('FailureReason')}")
            except Exception as e:
                print(f"Could not describe job {job_name}: {e}")

            try:
                # List all log streams
                streams = logs_client.describe_log_streams(
                    logGroupName=log_group_name,
                    logStreamNamePrefix=job_name
                )["logStreams"]
                
                if not streams:
                    print(f"No log streams found for job: {job_name}")
                    # Try without prefix to see all streams
                    streams = logs_client.describe_log_streams(
                        logGroupName=log_group_name
                    )["logStreams"]
                    
                    # Filter manually
                    streams = [s for s in streams if job_name in s["logStreamName"]]
                
                for stream in streams:
                    stream_name = stream["logStreamName"]
                    print(f"\n[Log Stream: {stream_name}]")
                    
                    # Get more than 50 lines for better debugging
                    try:
                        events = logs_client.get_log_events(
                            logGroupName=log_group_name,
                            logStreamName=stream_name,
                            limit=100  # Fetch last 100 log lines
                        )["events"]
                        
                        # Print recent errors first
                        for event in reversed(events):  # Show most recent first
                            message = event["message"]
                            if "error" in message.lower() or "exception" in message.lower() or "fail" in message.lower():
                                print(f"ERROR: {message}")
                            else:
                                print(message)
                    except Exception as e:
                        print(f"Could not get events from {stream_name}: {e}")
                        
            except Exception as e:
                print(f"Failed to fetch logs from CloudWatch: {e}")
                print(f"Check if the log group '{log_group_name}' exists and you have permissions")

In [70]:
def poll_pipeline_execution(execution_arn, sleep_seconds=30):
    """
    Poll a SageMaker Pipeline execution until it reaches a terminal status.
    """
    terminal_statuses = ["Succeeded", "Failed", "Stopped"]

    while True:
        desc = sm.describe_pipeline_execution(PipelineExecutionArn=execution_arn)

        status = desc["PipelineExecutionStatus"]

        print("=" * 80)
        print("Pipeline status:", status)
        print("Execution display name:", desc.get("PipelineExecutionDisplayName"))
        print("Start time:", desc.get("CreationTime") or desc.get("StartTime"))

        steps_response = sm.list_pipeline_execution_steps(
            PipelineExecutionArn=execution_arn
        )

        steps = steps_response.get("PipelineExecutionSteps", [])

        if steps:
            print("\nSteps:")
            for step in reversed(steps):
                step_name = step.get("StepName")
                step_status = step.get("StepStatus")
                failure_reason = step.get("FailureReason", "")

                print(f"- {step_name}: {step_status}")

                job_name = None
                log_group_name = None
            
                if failure_reason:
                    print(f"  Failure reason: {failure_reason}")
                    print_failed_step_logs(execution_arn)
        else:
            print("\nNo steps listed yet.")

        if status in terminal_statuses:
            print()
            print("Final status:", status)
            return status

        print()
        print(f"Still running. Checking again in {sleep_seconds} seconds...")
        time.sleep(sleep_seconds)


In [71]:
# Poll a pipeline execution
manual_status = poll_pipeline_execution(
    TRIGGERED_EXECUTION_ARN,
    sleep_seconds=30
)

Pipeline status: Executing
Execution display name: manual-triggered-test-1787481909
Start time: 2026-08-23 10:45:09.590000+00:00

Steps:
- PreprocessData: Executing

Still running. Checking again in 30 seconds...


Pipeline status: Executing
Execution display name: manual-triggered-test-1787481909
Start time: 2026-08-23 10:45:09.590000+00:00

Steps:
- PreprocessData: Executing

Still running. Checking again in 30 seconds...


Pipeline status: Executing
Execution display name: manual-triggered-test-1787481909
Start time: 2026-08-23 10:45:09.590000+00:00

Steps:
- PreprocessData: Executing

Still running. Checking again in 30 seconds...


Pipeline status: Executing
Execution display name: manual-triggered-test-1787481909
Start time: 2026-08-23 10:45:09.590000+00:00

Steps:
- PreprocessData: Executing

Still running. Checking again in 30 seconds...


Pipeline status: Executing
Execution display name: manual-triggered-test-1787481909
Start time: 2026-08-23 10:45:09.590000+00:00

Steps:
- PreprocessData: Executing

Still running. Checking again in 30 seconds...


...

Pipeline status: Executing
Execution display name: manual-triggered-test-1787481909
Start time: 2026-08-23 10:45:09.590000+00:00

Steps:
- PreprocessData: Succeeded
- TrainModel: Succeeded
- EvaluateModel: Executing

Still running. Checking again in 30 seconds...


Pipeline status: Executing
Execution display name: manual-triggered-test-1787481909
Start time: 2026-08-23 10:45:09.590000+00:00

Steps:
- PreprocessData: Succeeded
- TrainModel: Succeeded
- EvaluateModel: Executing

Still running. Checking again in 30 seconds...


Pipeline status: Executing
Execution display name: manual-triggered-test-1787481909
Start time: 2026-08-23 10:45:09.590000+00:00

Steps:
- PreprocessData: Succeeded
- TrainModel: Succeeded
- EvaluateModel: Executing

Still running. Checking again in 30 seconds...


Pipeline status: Executing
Execution display name: manual-triggered-test-1787481909
Start time: 2026-08-23 10:45:09.590000+00:00

Steps:
- PreprocessData: Succeeded
- TrainModel: Succeeded
- EvaluateModel: Executing

Still running. Checking again in 30 seconds...


Pipeline status: Succeeded
Execution display name: manual-triggered-test-1787481909
Start time: 2026-08-23 10:45:09.590000+00:00

Steps:
- PreprocessData: Succeeded
- TrainModel: Succeeded
- EvaluateModel: Succeeded
- CheckAUCQualityGate: Succeeded
- RegisterModel-RegisterModel: Succeeded

Final status: Succeeded


In [72]:
# Give EventBridge/Lambda a short time to start the pipeline execution.
time.sleep(10)

response = sm.list_pipeline_executions(
    PipelineName=TRIGGERED_PIPELINE_NAME,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1,
)

if not response["PipelineExecutionSummaries"]:
    raise RuntimeError(f"No pipeline executions found for {TRIGGERED_PIPELINE_NAME}")

latest_execution_arn = response["PipelineExecutionSummaries"][0]["PipelineExecutionArn"]

print("Latest pipeline execution:")
print(latest_execution_arn)
print()

trigger_status = poll_pipeline_execution(latest_execution_arn, sleep_seconds=30)


Latest pipeline execution:
arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team09-bank-fraud-detection-triggered/execution/xhfheuuda6y0

Pipeline status: Succeeded
Execution display name: manual-triggered-test-1787481909
Start time: 2026-08-23 10:45:09.590000+00:00

Steps:
- PreprocessData: Succeeded
- TrainModel: Succeeded
- EvaluateModel: Succeeded
- CheckAUCQualityGate: Succeeded
- RegisterModel-RegisterModel: Succeeded

Final status: Succeeded
